In [1]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import PowerTransformer, QuantileTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif
from sklearn.naive_bayes import CategoricalNB, GaussianNB
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.ensemble import (
    GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier,
    BaggingClassifier, StackingClassifier, HistGradientBoostingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold

train = pd.read_csv("train.csv")
X = train.drop(columns=["class"])
y = train["class"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate(model, X, y):
    scores = cross_validate(
        model, X, y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True
    )
    val_mean  = scores["test_score"].mean()
    val_std   = scores["test_score"].std()
    train_mean = scores["train_score"].mean()
    return val_mean, val_std, train_mean

# ─── MODELOS ──────────────────────────────────────────────────────────────────
modelos = {
    "LDA": Pipeline([
        ("scaler",   PowerTransformer()),
        ("selector", SelectKBest(score_func=mutual_info_classif, k=20)),
        ("lda",      LinearDiscriminantAnalysis(solver="lsqr", shrinkage=0.3))
    ]),
    "CategoricalNB": make_pipeline(
        KBinsDiscretizer(n_bins=30, encode="ordinal", strategy="quantile"),
        SelectKBest(f_classif, k=60),
        CategoricalNB()
    ),
    "GaussianNB": make_pipeline(
        QuantileTransformer(output_distribution="normal", random_state=42),
        SelectKBest(f_classif, k=50),
        GaussianNB()
    ),
    "AdaBoost": make_pipeline(
        SelectKBest(f_classif, k=30),
        AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=2),
            n_estimators=500,
            learning_rate=0.1,
            random_state=42
        )
    ),
    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=15, random_state=42),
        n_estimators=100,
        max_samples=0.65,
        max_features=0.1,
        random_state=42,
        n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        max_features="sqrt",
        min_samples_leaf=10,
        min_samples_split=5,
        max_samples=1.0,
        criterion="gini",
        random_state=42,
        bootstrap=True
    ),
    "Gradient Boosting": Pipeline([
        ("selector", SelectKBest(f_classif, k=50)),
        ("model",    GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.5,
            min_samples_leaf=3,
            random_state=42
        ))
    ]),
    "Stacking": StackingClassifier(
        estimators=[
            ("lda", Pipeline([
                ("scaler", PowerTransformer()),
                ("pca",    PCA(n_components=50)),
                ("lda",    LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))
            ])),
            ("hgb", HistGradientBoostingClassifier(
                learning_rate=0.08,
                max_iter=120
            ))
        ],
        final_estimator=LogisticRegression(C=0.5),
        cv=5
    ),
}

# ─── MEJORES PARÁMETROS (resumen legible) ─────────────────────────────────────
best_params = {
    "LDA":               "PowerTransformer | SelectKBest(MI, k=20) | shrinkage=0.3",
    "CategoricalNB":     "KBins(n_bins=30) | SelectKBest(k=60)",
    "GaussianNB":        "QuantileTransformer | SelectKBest(k=50)",
    "AdaBoost":          "SelectKBest(k=30) | n_estimators=500, lr=0.1, depth=2",
    "Bagging":           "depth=15, n=100, max_samples=0.65, max_features=0.1",
    "Random Forest":     "n=500, max_features=sqrt, min_leaf=10, min_split=5",
    "Gradient Boosting": "SelectKBest(k=50) | n=150, lr=0.05, depth=4, subsample=0.5",
    "Stacking":          "LDA(PCA=50) + HGB(lr=0.08, iter=120) | meta: LR(C=0.5)",
}

# ─── EVALUACIÓN ───────────────────────────────────────────────────────────────
resultados = []
for nombre, modelo in modelos.items():
    mean, std, train = evaluate(modelo, X, y)
    resultados.append({
        "Modelo":          nombre,
        "Mejores parámetros": best_params[nombre],
        "Train":           round(train, 4),
        "CV accuracy":     round(mean, 4),
        "± std":           round(std, 4),
    })

df_final = pd.DataFrame(resultados)
df_final = df_final.sort_values("CV accuracy", ascending=False).reset_index(drop=True)
display(df_final)

c:\Users\laura\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
c:\Users\laura\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\laura\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 13 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\laura\AppData\Local\P

,Modelo,Mejores parámetros,Train,CV accuracy,± std
0,Random Forest,"n=500, max_features=sqrt, min_leaf=10, min_spl...",0.9438,0.829,0.0185
1,Bagging,"depth=15, n=100, max_samples=0.65, max_feature...",0.9995,0.823,0.0075
2,Gradient Boosting,"SelectKBest(k=50) | n=150, lr=0.05, depth=4, s...",0.9998,0.819,0.0203
3,CategoricalNB,KBins(n_bins=30) | SelectKBest(k=60),0.8982,0.816,0.0183
4,Stacking,"LDA(PCA=50) + HGB(lr=0.08, iter=120) | meta: L...",0.9995,0.813,0.0218
5,AdaBoost,"SelectKBest(k=30) | n_estimators=500, lr=0.1, ...",0.8918,0.802,0.0378
6,GaussianNB,QuantileTransformer | SelectKBest(k=50),0.7992,0.793,0.0209
7,LDA,"PowerTransformer | SelectKBest(MI, k=20) | shr...",0.7642,0.763,0.0238
